In [105]:
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from datetime import datetime
import zipfile
from dotenv import load_dotenv

In [106]:
# setup arguments
load_dotenv()

username = os.getenv("MY_USERNAME")
password = os.getenv("MY_PASSWORD")
url = "https://the-internet.herokuapp.com/login"
url_download = "https://the-internet.herokuapp.com/download"

chrome_options = Options()
parent_dir = os.path.dirname(os.getcwd())
print(parent_dir)

download_path = os.path.join(parent_dir, "tmp")

print(f"Path ใหม่: {download_path}")

chrome_options = Options()

# 2. add Prefs
prefs = {
    "download.default_directory": download_path, # ต้องเป็น Absolute Path
    "download.prompt_for_download": False,        # ห้ามถามที่เซฟ
    "download.directory_upgrade": True,
    "safebrowsing.enabled": False,                # ปิด Safe Browsing ชั่วคราว (เพื่อไม่ให้มันกักไฟล์)
    "plugins.always_open_pdf_externally": True,  # ปิดการเปิดไฟล์ในเบราว์เซอร์
    "profile.default_content_settings.popups": 0, # ปิด Popup ทุกชนิด
}
chrome_options.add_experimental_option("prefs", prefs)

# 3. เพิ่ม Arguments เพื่อปิดระบบความปลอดภัยที่ขวางการโหลด
chrome_options.add_argument("--disable-features=InsecureDownloadWarnings")
chrome_options.add_argument("--ignore-certificate-errors")


c:\Users\Kanok Onteaun\Desktop\TRUE\Project\MNP_Dashboard
Path ใหม่: c:\Users\Kanok Onteaun\Desktop\TRUE\Project\MNP_Dashboard\tmp


In [107]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

# 2. ไปหน้า Login
driver.get(url)
    
# 3. สั่ง Login (ดึง element จาก id)
driver.find_element(By.ID, "username").send_keys(username)
driver.find_element(By.ID, "password").send_keys(password)
driver.find_element(By.CSS_SELECTOR, "button[type='submit']").click()
time.sleep(5) # รอให้ดาวน์โหลดลงเครื่องเสร็จ

print("Login สำเร็จ: ", driver.find_element(By.TAG_NAME, "h2").text)

try:
    driver.get(url_download)
    
    # 3. สั่งคลิกด้วย JavaScript (เพื่อความชัวร์ว่าจะไม่ติด Popup บัง)
    element = driver.find_element(By.LINK_TEXT, "sample-zip-file.zip")
    driver.execute_script("arguments[0].click();", element)
    print(f"เริ่มดาวน์โหลด {target_file}...")

    # 4. ฟังก์ชันรอจนกว่าจะโหลดเสร็จ
    def wait_for_download(directory, timeout=60):
        seconds = 0
        while seconds < timeout:
            # เช็คว่ามีไฟล์ที่นามสกุลลงท้ายด้วย .crdownload หรือ .tmp ไหม
            files = os.listdir(directory)
            if any(f.endswith('.crdownload') or f.endswith('.tmp') for f in files):
                time.sleep(1)
                seconds += 1
            else:
                # ถ้าไม่มีไฟล์ชั่วคราวแล้ว และมีไฟล์เป้าหมายปรากฏขึ้นมา
                if target_file in os.listdir(directory):
                    return True
                time.sleep(1)
                seconds += 1
        return False

    if wait_for_download(download_path):
        print("✅ ดาวน์โหลดสำเร็จ! ไฟล์เปลี่ยนจาก .tmp เป็น .zip เรียบร้อย")
    else:
        print("❌ การดาวน์โหลดใช้เวลานานเกินไป หรือเกิดข้อผิดพลาด")

finally:
    # อย่าเพิ่งรีบปิดถ้ายังไม่แน่ใจ แต่ใน try-finally จะช่วยปิด driver ให้สะอาด
    driver.quit()

Login สำเร็จ:  Secure Area
เริ่มดาวน์โหลด sample-zip-file.zip...
✅ ดาวน์โหลดสำเร็จ! ไฟล์เปลี่ยนจาก .tmp เป็น .zip เรียบร้อย


In [108]:
zip_file_path = os.path.join(download_path, "sample-zip-file.zip") 
file_password = "your_password_here"

try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        # การใส่รหัสผ่านใน zipfile ต้องแปลงเป็น bytes โดยใช้ .encode()
        zip_ref.extractall(path=download_path)
    print(f"✅ แตกไฟล์ลงใน {download_path} เรียบร้อยแล้ว")

except RuntimeError as e:
    if "password" in str(e):
        print("❌ รหัสผ่านไม่ถูกต้อง")
    else:
        print(f"❌ เกิดข้อผิดพลาดขณะแตกไฟล์: {e}")
except zipfile.BadZipFile:
    print("❌ ไฟล์ ZIP เสียหรือโหลดมาไม่สมบูรณ์")
os.remove(zip_file_path)




✅ แตกไฟล์ลงใน c:\Users\Kanok Onteaun\Desktop\TRUE\Project\MNP_Dashboard\tmp เรียบร้อยแล้ว


In [104]:
# 1. ระบุชื่อไฟล์ที่ต้องการอ่าน (ตรวจสอบให้แน่ใจว่าชื่อตรงกับใน ZIP)
file_to_read = "sample.txt"
file_path = os.path.join(download_path, file_to_read)

# 2. เริ่มการอ่านไฟล์
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read() # อ่านเนื้อหาทั้งหมด
        print("--- เนื้อหาในไฟล์ ---")
        print(content)
        print("--------------------")
except FileNotFoundError:
    print(f"❌ หาไฟล์ไม่เจอที่: {file_path}")
except UnicodeDecodeError:
    # หากไฟล์เป็นภาษาไทยแล้วอ่านไม่ได้ ให้ลองเปลี่ยน encoding เป็น 'cp874' หรือ 'tis-620'
    with open(file_path, 'r', encoding='cp874') as f:
        print(f.read())

--- เนื้อหาในไฟล์ ---
I would love to try or hear the sample audio your app can produce. I do not want to purchase, because I've purchased so many apps that say they do something and do not deliver.  

Can you please add audio samples with text you've converted? I'd love to see the end results.

Thanks!
--------------------
